# analyse01 — Data Exploration (Week 1)

**Goal:** Get oriented in the Backblaze hard drive dataset using DuckDB, without loading
the full dataset into memory. The goal is to identify:

- What columns / types we have
- How many rows / drives / days we're working with
- Which drive models actually have enough failure events to analyze
- Framing target research question


In [1]:
import os
print("Current working dir:", os.getcwd())
print("Files in dir", os.listdir())

Current working dir: C:\Users\User\Documents\Personal_Portfolios\Backblaze_Hard_Drive\Hard-Drive-Reliability-Analysis\notebooks
Files in dir ['.ipynb_checkpoints', '01_data_exploration.ipynb', '01_edit.ipynb']


In [2]:
# %pip install duckdb jupysql duckdb-engine

In [3]:
import sys
sys.path.append("../src")
import load_data as ld

con = ld.get_connection()

DATA_DIR = "../data/raw"

## 1. Schema Check
What columns do we actually have, and what types did DuckDB infer?

In [4]:
ld.schema(con, DATA_DIR)

,column_name,column_type,null,key,default,extra
0,date,DATE,YES,None,None,None
1,serial_number,VARCHAR,YES,None,None,None
2,model,VARCHAR,YES,None,None,None
3,capacity_bytes,BIGINT,YES,None,None,None
4,failure,BIGINT,YES,None,None,None
...,...,...,...,...,...,...
192,smart_252_raw,VARCHAR,YES,None,None,None
193,smart_254_normalized,BIGINT,YES,None,None,None
194,smart_254_raw,BIGINT,YES,None,None,None
195,smart_255_normalized,VARCHAR,YES,None,None,None


**Note:** Backblaze's CSVs have ~130+ SMART columns (many mostly null,
since not all drive models report all attributes). 
During data cleaning, check which SMART columns are almost entirely null to drop rather than trying to impute.


## 2. Row Count and Date Range
Sanity check (all 90 files, full quarter).

In [5]:
print("Total drive-day rows:", ld.row_count(con, DATA_DIR))
ld.date_range(con, DATA_DIR)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Total drive-day rows: 27799986


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,min_date,max_date,n_days
0,2025-01-01,2025-03-31,90


## 3. Drive Models Present
How many distinct drives and rows per model? Which models are worth analysing?

In [6]:
ld.unique_models(con, DATA_DIR)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,model,n_drives,n_rows
0,TOSHIBA MG08ACA16TA,40273,3616128
1,TOSHIBA MG07ACA14TA,37705,3385931
2,ST16000NM001G,33817,3029485
3,WDC WUH722222ALE6L4,34857,2870372
4,WDC WUH721816ALE6L4,26428,2375575
...,...,...,...
71,Samsung SSD 850 EVO 1TB,1,90
72,SSDSCKKB240GZR,1,90
73,ST1000LM024 HN,1,90
74,Samsung SSD 850 PRO 1TB,1,24


## 4. Failure Summary by Model

This is the number that determines project scope. Failures are rare events, so a model
with only 1-2 failure events in the whole quarter would not support any meaningful
statistical claim, no matter how many healthy drive-days it has.

In [7]:
ld.failure_summary(con, DATA_DIR)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,model,drive_days,failures,failure_rate_pct
0,TOSHIBA MG07ACA14TA,3385931,133.0,0.0039
1,HGST HUH721212ALN604,911546,124.0,0.0136
2,ST8000NM0055,1212798,109.0,0.0090
3,ST12000NM0008,1714395,104.0,0.0061
4,TOSHIBA MG08ACA16TA,3616128,94.0,0.0026
...,...,...,...,...
71,ST14000NM002J,360,0.0,0.0000
72,Seagate SSD,8996,0.0,0.0000
73,WDC WD5000LPVX,5092,0.0,0.0000
74,ST12000NM003G,840,0.0,0.0000


## 5. Scope Decision: Which Models make the cut?

Set a minimum failure-event threshold to see what survives. Adjust `MIN_FAILURES` based 
on the real numbers. A common rule of thumb is to want at least ~20-30 failure events
per group before trusting a comparison.

In [8]:
ld.threshold_sensitivity(con, DATA_DIR, thresholds=[10, 20, 30, 40, 50, 60, 100])

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,Y,n_models_eligible,failures_covered,pct_failures_covered
0,10,20,1046.0,98.0
1,20,16,993.0,93.1
2,30,11,877.0,82.2
3,40,9,805.0,75.4
4,50,8,764.0,71.6
5,60,6,657.0,61.6
6,100,4,470.0,44.0


In the first consecutive increase in % failures covered, the largest drop are at Y=20->Y=30. Determine that elbow is at Y=30 may be reasonable.

In [9]:
MIN_FAILURES = 30
eligible = ld.drives_with_enough_failures(con, DATA_DIR, min_failures=MIN_FAILURES)
eligible

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,model,drive_days,failures,failure_rate_pct
0,TOSHIBA MG07ACA14TA,3385931,133.0,0.0039
1,HGST HUH721212ALN604,911546,124.0,0.0136
2,ST8000NM0055,1212798,109.0,0.0090
3,ST12000NM0008,1714395,104.0,0.0061
4,TOSHIBA MG08ACA16TA,3616128,94.0,0.0026
5,HGST HUH721212ALE604,1197864,93.0,0.0078
6,WDC WUH722222ALE6L4,2870372,56.0,0.0020
7,ST8000DM002,814279,51.0,0.0063
8,ST16000NM001G,3029485,41.0,0.0014
9,ST14000NM001G,951836,37.0,0.0039


## 6. Target Variable

**Decision:** Drive-level binary classification — "ever failed vs. never failed in the quarter."

Changes the **unit of analysis** from drive per day to drive state. The raw data
has one row per drive per day (a drive present for 90 days = 90 rows). Therefore,
each unique `serial_number` collapses into a single row:

- **label = 1** if the drive has any `failure = 1` record anywhere in the quarter
- **label = 0** if the drive appears in the data but never fails

**Feature snapshot rule:** since SMART attributes are daily readings, then only one
representative reading per drive is needed, not 90 individual readings. 
- For failed drives, "most recent" = the reading right before the failure day.
- For healthy drives, "most recent" = last day in the dataset (should be at
  the end of the quarter, unless the drive was somehow replaced).

**Alternative considered:** aggregate stats (e.g., max value per attribute) over each
drive's full history instead of a single snapshot. Since SMART attributes 
(e.g. reallocated sector count) can be cumulative, so max ≈ last
reading anyway, therefore rejected. In conclusion, the single-snapshot rule is simpler 
and gives the same signal for those attributes, at lower complexity.
                                                                               

## 7. Research Question

> "Among drive models with sufficient failure events (n >= [30] failures), which SMART
> attributes — measured at each drive's most recent reading in the quarter — are most
> associated with whether that drive ever failed during Q1 2025?"
